In [ ]:
%%bash

# --- 1. PASTE YOUR HUGGING FACE TOKEN HERE ---
HF_TOKEN="xxxxxxxxxxxxxxxxxxxx" # <--- REPLACE WITH YOUR REAL TOKEN

# --- 2. FAIL-FAST & CLEANUP ---
set -e
echo "🧹 Cleaning up workspace..."
cd /kaggle/working/
rm -rf sam3

# --- 3. INSTALL DEPENDENCIES ---
echo "📦 Installing PyTorch and dependencies..."
pip install -q --upgrade pip setuptools wheel
pip install -q torch==2.7.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# --- 4. CLONE & INSTALL SAM 3 ---
echo "⬇️ Cloning and installing SAM 3..."
git clone https://github.com/facebookresearch/sam3.git > /dev/null 2>&1
cd sam3
pip install -q -e .
pip install -q -e ".[notebooks]"

# --- 5. FIX NUMPY/NUMBA INCOMPATIBILITY ---
echo "🔧 Fixing NumPy/Numba conflict..."
pip install -q --force-reinstall numpy==1.26.4 numba

# --- 6. AUTHENTICATE & DOWNLOAD VOCAB (NON-INTERACTIVE) ---
echo "🔑 Authenticating with Hugging Face..."
huggingface-cli login --token $HF_TOKEN --add-to-git-credential > /dev/null 2>&1

echo "⬇️ Downloading missing BPE vocab file..."
mkdir -p assets
wget -q -O assets/bpe_simple_vocab_16e6.txt.gz https://openaipublic.azureedge.net/clip/bpe_simple_vocab_16e6.txt.gz

echo ""
echo "🎉 Environment setup is 100% complete and correct."

🧹 Cleaning up workspace...
📦 Installing PyTorch and dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.0 MB/s eta 0:00:00
⬇️ Cloning and installing SAM 3...
🔧 Fixing NumPy/Numba conflict...
🔑 Authenticating with Hugging Face...
⬇️ Downloading missing BPE vocab file...

🎉 Environment setup is 100% complete and correct.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.2.19 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2

In [2]:
%%writefile generate_final.py

import os
import torch
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
from PIL import Image
import json
import numba
from tqdm import tqdm
import torch.distributed as dist
import torch.multiprocessing as mp

# Add sam3 to Python path
import sys
if "/kaggle/working/sam3" not in sys.path:
    sys.path.append("/kaggle/working/sam3")

from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
class Config:
    INPUT_DATA_ROOT = "/kaggle/input/coco-2017-dataset/coco2017"
    OUTPUT_DIR = "/kaggle/working/synthetic_forgery_dataset"
    
    # --- BATCH CONTROL (NEW) ---
    # Set these to process chunks. 
    # Example: Run 1 -> Start=0, End=40000
    #          Run 2 -> Start=40000, End=80000
    START_INDEX = 0
    END_INDEX =   28000
    
    SMOKE_TEST = False # Set to False for full run
    SMOKE_TEST_SIZE = 10
    
    MAX_DIMENSION = 1024
    CONFIDENCE_THRESHOLD = 0.4
    MIN_CANDIDATES = 1
    MAX_CANDIDATES = 8
    
    NUM_OBJECTS_TO_FORGE = (1, 6)
    COPIES_PER_OBJECT = (1, 3)
    
    FOLDER_PROMPTS = {
        'test2017': ['object', 'animal', 'human','thing']
    }
    
    # --- PROBABILITY DISTRIBUTIONS ---
    INSTANCE_COUNTS = [1, 2, 3, 4, 5, 6]
    INSTANCE_PROBS =  [0.30, 0.25, 0.20, 0.12, 0.08, 0.05]
    
    COPY_COUNTS = [1, 2, 3, 4]
    COPY_PROBS =  [0.65, 0.25, 0.08, 0.02]
    
    ROTATE_RANGE = (-180, 180)
    SCALE_RANGE = (0.8, 1.2)
    MAX_OVERLAP_IOU = 0.05

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
import numpy.typing as npt

@numba.jit(nopython=True)
def _rle_encode_jit(x, fg_val=1):
    dots = np.where(x.T.flatten() == fg_val)[0]
    run_lengths = []; prev = -2
    for b in dots:
        if b > prev + 1: run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1; prev = b
    return run_lengths

def rle_encode(masks, fg_val=1):
    return ';'.join([json.dumps(_rle_encode_jit(x, fg_val)) for x in masks])

@numba.njit
def _rle_decode_jit(mask_rle, height, width):
    if len(mask_rle) % 2 != 0: raise ValueError('Odd number of values.')
    starts, lengths = mask_rle[0::2], mask_rle[1::2]; starts -= 1
    ends = starts + lengths
    for i in range(len(starts) - 1):
        if ends[i] > starts[i + 1]: raise ValueError('Overlapping pixels.')
    img = np.zeros(height * width, dtype=np.bool_)
    for lo, hi in zip(starts, ends): img[lo:hi] = 1
    return img

def rle_decode(mask_rle, shape):
    mask_rle = json.loads(mask_rle); mask_rle = np.asarray(mask_rle, dtype=np.int32)
    return _rle_decode_jit(mask_rle, shape[0], shape[1]).reshape(shape, order='F')

def resize_keeping_aspect_ratio(image, max_dim):
    w, h = image.size
    if max(w, h) <= max_dim: return image, w, h
    scale = max_dim / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    return image.resize((new_w, new_h), Image.Resampling.LANCZOS), new_w, new_h

def affine_transform(img, mask, force_shrink=False):
    y_idx, x_idx = np.where(mask > 0);
    if len(y_idx) == 0: return None, None
    y1, y2, x1, x2 = y_idx.min(), y_idx.max(), x_idx.min(), x_idx.max()
    crop_img, crop_mask = img[y1:y2+1, x1:x2+1], mask[y1:y2+1, x1:x2+1]
    h, w = crop_img.shape[:2]; center = (w//2, h//2)
    angle = random.uniform(*Config.ROTATE_RANGE)
    scale = random.uniform(0.5, 0.75) if force_shrink else random.uniform(*Config.SCALE_RANGE)
    M = cv2.getRotationMatrix2D(center, angle, scale)
    nw, nh = int((h*np.abs(M[0,1]))+(w*np.abs(M[0,0]))), int((h*np.abs(M[0,0]))+(w*np.abs(M[0,1])))
    M[0, 2] += (nw/2)-center[0]; M[1, 2] += (nh/2)-center[1]
    res_img = cv2.warpAffine(crop_img, M, (nw, nh), borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))
    res_mask = cv2.warpAffine(crop_mask, M, (nw, nh), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return res_img, res_mask

def calculate_iou(mask1, mask2):
    return np.sum(np.logical_and(mask1, mask2)) / (np.sum(np.logical_or(mask1, mask2)) + 1e-6)

# ==============================================================================
# 3. CORE LOGIC
# ==============================================================================
def find_candidates_iteratively(processor, pil_img, folder_name, img_w, img_h):
    prompts = list(Config.FOLDER_PROMPTS.get(folder_name, ["object"])); random.shuffle(prompts)
    candidates, seen_masks = [], []
    for prompt in prompts:
        if len(candidates) >= Config.MAX_CANDIDATES: break
        with torch.autocast("cuda", dtype=torch.float16):
            state = processor.set_image(pil_img); processor.reset_all_prompts(state)
            state = processor.set_text_prompt(state=state, prompt=prompt)
        masks = state.get('pred_masks', state.get('masks', None))
        if masks is None: continue
        if isinstance(masks, torch.Tensor): masks = masks.detach().cpu().float().numpy()
        if masks.ndim == 4: masks = masks.squeeze(1)
        for m in masks:
            m_bin = (m > 0).astype(np.uint8); area = np.sum(m_bin)
            if not (50 < area < (img_w * img_h * 0.7)): continue
            if any(calculate_iou(m_bin, seen) > 0.5 for seen in seen_masks): continue
            candidates.append(m_bin); seen_masks.append(m_bin)
            if len(candidates) >= Config.MAX_CANDIDATES: break
    return candidates

def generate_and_save_sample(processor, img_path, folder_name, out_img_dir, out_mask_dir, out_vis_dir):
    try:
        pil_img = Image.open(img_path).convert("RGB")
        pil_img, W, H = resize_keeping_aspect_ratio(pil_img, Config.MAX_DIMENSION)
        forged_np, gt_mask = np.array(pil_img), np.zeros((H, W), dtype=np.uint8)
        
        # 1. Find Candidates
        candidates = find_candidates_iteratively(processor, pil_img, folder_name, W, H)
        if len(candidates) < Config.MIN_CANDIDATES: return False

        # 2. Smart Logic: Count
        num_instances = random.choices(Config.INSTANCE_COUNTS, weights=Config.INSTANCE_PROBS, k=1)[0]
        num_instances = min(num_instances, len(candidates))
        selected_masks = random.sample(candidates, num_instances)
        
        # 3. Setup Occupancy
        occupancy_map = np.zeros((H, W), dtype=np.uint8)
        for cand in candidates: occupancy_map[cand > 0] = 1
        
        successful_pastes = 0
        
        # 4. Forgery Loop
        for i, mask_patch in enumerate(selected_masks):
            inst_id = i + 1
            obj_img = forged_np.copy(); obj_img[mask_patch == 0] = 0
            
            copies_target = random.choices(Config.COPY_COUNTS, weights=Config.COPY_PROBS, k=1)[0]
            
            for _ in range(copies_target):
                placed = False
                for shrink in [False, True]:
                    if placed: break
                    for _ in range(25):
                        res_img, res_mask = affine_transform(obj_img, mask_patch, force_shrink=shrink)
                        if res_img is None: break
                        sh, sw = res_img.shape[:2]
                        if sh >= H or sw >= W: continue
                        py, px = random.randint(0, H - sh), random.randint(0, W - sw)
                        
                        # Overlap Check
                        roi_occ = occupancy_map[py:py+sh, px:px+sw]
                        roi_mask_bool = res_mask > 0
                        if (np.sum(np.logical_and(roi_mask_bool, roi_occ > 0)) / (np.sum(roi_mask_bool) + 1e-6)) > Config.MAX_OVERLAP_IOU:
                            continue
                        
                        # Paste
                        roi = forged_np[py:py+sh, px:px+sw]
                        roi_mask = gt_mask[py:py+sh, px:px+sw]
                        if roi.shape[:2] != res_img.shape[:2]: continue
                        
                        roi[roi_mask_bool] = res_img[roi_mask_bool]
                        roi_mask[roi_mask_bool] = inst_id
                        forged_np[py:py+sh, px:px+sw] = roi
                        gt_mask[py:py+sh, px:px+sw] = roi_mask
                        
                        # Update Occupancy
                        occupancy_map[py:py+sh, px:px+sw] = np.logical_or(roi_occ, roi_mask_bool)
                        gt_mask[mask_patch > 0] = inst_id 
                        
                        placed = True; successful_pastes += 1; break
        
        if successful_pastes == 0: return False

        base_name = f"{folder_name}_{os.path.splitext(os.path.basename(img_path))[0]}"
        
        # Save Forged Image
        Image.fromarray(forged_np).save(os.path.join(out_img_dir, f"{base_name}.png"))
        
        # Save RLE Mask
        instance_ids = np.unique(gt_mask)[1:]
        binary_masks = [(gt_mask == i).astype(np.uint8) for i in instance_ids]
        np.save(os.path.join(out_mask_dir, f"{base_name}.npy"), rle_encode(binary_masks))
        
        # --- NEW: SAVE VISUALIZED MASK ---
        vis_mask = np.zeros((H, W, 3), dtype=np.uint8)
        for uid in instance_ids:
            # Deterministic color generation based on ID
            # Uses a fixed seed calculation so ID 1 is always same color, ID 2 same, etc.
            # But varies enough to distinguish adjacent IDs
            r = (uid * 123 + 50) % 255
            g = (uid * 45 + 100) % 255
            b = (uid * 77 + 150) % 255
            vis_mask[gt_mask == uid] = [r, g, b]
            
        Image.fromarray(vis_mask).save(os.path.join(out_vis_dir, f"{base_name}.png"))
        
        return True
    except Exception: return False

# ==============================================================================
# DDP WORKER & LAUNCHER
# ==============================================================================
def run_worker(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'; os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    
    if rank == 0: print(f"🚀 Loading SAM 3 on GPU {rank}...")
    base_dir = "/kaggle/working/sam3"
    vocab_path = os.path.join(base_dir, "assets/bpe_simple_vocab_16e6.txt.gz")
    model = build_sam3_image_model(bpe_path=vocab_path).to(f'cuda:{rank}')
    processor = Sam3Processor(model, confidence_threshold=Config.CONFIDENCE_THRESHOLD)
    
    # 1. GATHER ALL PATHS
    all_paths = []
    for folder in Config.FOLDER_PROMPTS.keys():
        folder_path = os.path.join(Config.INPUT_DATA_ROOT, folder)
        if os.path.exists(folder_path):
            all_paths.extend([(os.path.join(folder_path, f), folder) for f in os.listdir(folder_path) if f.lower().endswith(('png', 'jpg'))])
    
    # 2. SORT FOR DETERMINISTIC SLICING (CRITICAL!)
    # Without this, "0 to 40k" might mean different images in different runs
    all_paths.sort()
    
    # 3. APPLY BATCH SLICING
    # This selects the global chunk we want to process in this run
    if Config.END_INDEX is not None:
        batch_paths = all_paths[Config.START_INDEX : Config.END_INDEX]
    else:
        batch_paths = all_paths[Config.START_INDEX :]
        
    if rank == 0:
        print(f"📊 Total Dataset Size: {len(all_paths)}")
        print(f"✂️ Processing Chunk: {Config.START_INDEX} to {Config.END_INDEX} (Size: {len(batch_paths)})")
    
    # 4. SPLIT FOR DDP
    # Now distribute this chunk among the GPUs
    if Config.SMOKE_TEST:
        paths_to_process = random.sample(batch_paths, min(len(batch_paths), Config.SMOKE_TEST_SIZE))
    else:
        paths_to_process = batch_paths
        
    my_paths = paths_to_process[rank::world_size]
    
    # Paths for 3 output folders
    out_img_dir = os.path.join(Config.OUTPUT_DIR, "images")
    out_mask_dir = os.path.join(Config.OUTPUT_DIR, "masks")
    out_vis_dir = os.path.join(Config.OUTPUT_DIR, "mask_visualized")
    
    if rank == 0:
        os.makedirs(out_img_dir, exist_ok=True)
        os.makedirs(out_mask_dir, exist_ok=True)
        os.makedirs(out_vis_dir, exist_ok=True)
        
    for img_path, folder_name in tqdm(my_paths, desc=f"GPU {rank}", position=rank, disable=(rank != 0)):
        generate_and_save_sample(processor, img_path, folder_name, out_img_dir, out_mask_dir, out_vis_dir)
    
    dist.barrier()
    dist.destroy_process_group()

def verify_smoke_test():
    print("\n--- SMOKE TEST VERIFICATION ---")
    img_dir = os.path.join(Config.OUTPUT_DIR, "images")
    vis_dir = os.path.join(Config.OUTPUT_DIR, "mask_visualized")
    
    saved_images = [f for f in os.listdir(img_dir) if f.endswith('.png')]
    if not saved_images: print("❌ Verification failed."); return
    
    base_name = os.path.splitext(random.choice(saved_images))[0]
    img_path = os.path.join(img_dir, f"{base_name}.png")
    vis_path = os.path.join(vis_dir, f"{base_name}.png")
    
    verified_img = Image.open(img_path)
    verified_vis = Image.open(vis_path)
    
    plt.figure(figsize=(12, 6));
    plt.subplot(1, 2, 1); plt.imshow(verified_img); plt.title("Verified Forged Image")
    plt.subplot(1, 2, 2); plt.imshow(verified_vis); plt.title("Verified Color Mask (Visualized)")
    plt.show()

if __name__ == '__main__':
    mp.set_start_method('spawn', force=True)
    world_size = torch.cuda.device_count()
    if world_size > 1:
        mp.spawn(run_worker, args=(world_size,), nprocs=world_size, join=True)
    else:
        run_worker(0, 1)
    if Config.SMOKE_TEST:
        try: verify_smoke_test()
        except Exception as e: print(f"Smoke test verification failed: {e}")

Writing generate_final.py


In [3]:
# Run the Python script directly. It will detect the number of GPUs and spawn processes.
!python generate_final.py

/kaggle/working/sam3/sam3/model_builder.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' a

In [4]:
# # mask visualizer
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# from PIL import Image
# import json
# import random

# # --- 1. Define Paths ---
# OUTPUT_DIR = "/kaggle/working/synthetic_forgery_dataset"
# IMG_DIR = os.path.join(OUTPUT_DIR, "images")
# MASK_DIR = os.path.join(OUTPUT_DIR, "masks")

# # --- 2. Pure Python RLE Decoder (No Numba needed) ---
# def rle_decode(mask_rle_str, shape):
#     """
#     Decodes RLE string into a binary mask without Numba.
#     Safe for visualization.
#     """
#     try:
#         # The string contains JSON lists separated by semicolons
#         # Handle cases where it might be loaded as bytes or odd formats
#         if hasattr(mask_rle_str, 'item'): 
#             mask_rle_str = mask_rle_str.item()
            
#         # Parse JSON
#         mask_rle = json.loads(mask_rle_str)
        
#         # Create flat array
#         img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
        
#         # Loop through runs (Start, Length, Start, Length...)
#         # Note: Your encoder uses 1-based indexing for starts, so we subtract 1
#         for i in range(0, len(mask_rle), 2):
#             start = mask_rle[i] - 1
#             length = mask_rle[i+1]
#             img[start : start + length] = 1
            
#         return img.reshape(shape, order='F')
#     except Exception as e:
#         print(f"Error decoding RLE: {e}")
#         return np.zeros(shape, dtype=np.uint8)

# def visualize_generated_samples(num_samples=3):
#     if not os.path.exists(IMG_DIR):
#         print(f"❌ Directory not found: {IMG_DIR}")
#         return

#     files = [f for f in os.listdir(IMG_DIR) if f.endswith('.png')]
#     if not files:
#         print("❌ No images found. Did the generation script finish?")
#         return

#     print(f"✅ Found {len(files)} generated images. Showing {num_samples} samples...\n")
    
#     # Pick random samples
#     selected_files = random.sample(files, min(len(files), num_samples))

#     for fname in selected_files:
#         base_name = os.path.splitext(fname)[0]
#         img_path = os.path.join(IMG_DIR, fname)
#         mask_path = os.path.join(MASK_DIR, f"{base_name}.npy")

#         if not os.path.exists(mask_path):
#             print(f"⚠️ Missing mask for {fname}")
#             continue

#         # Load Image
#         image = Image.open(img_path)
#         w, h = image.size

#         # Load Mask
#         try:
#             rle_string = np.load(mask_path, allow_pickle=True)
#             # If it's a 0-d array wrapping the string
#             if rle_string.shape == ():
#                 rle_string = rle_string.item()
#             else:
#                 rle_string = str(rle_string)
#         except Exception as e:
#             print(f"Error loading npy: {e}")
#             continue

#         # Decode: The string is "RLE_Instance1;RLE_Instance2;..."
#         rle_parts = rle_string.split(';')
        
#         # Combine instances into one mask for visualization
#         combined_mask = np.zeros((h, w), dtype=np.uint8)
#         for i, rle in enumerate(rle_parts):
#             if not rle: continue
#             inst_mask = rle_decode(rle, (h, w))
#             # Assign Instance ID (Color)
#             combined_mask[inst_mask > 0] = i + 1  

#         # Plot
#         plt.figure(figsize=(12, 5))
        
#         plt.subplot(1, 2, 1)
#         plt.imshow(image)
#         plt.title(f"Forged Image: {base_name}")
#         plt.axis('off')

#         plt.subplot(1, 2, 2)
#         plt.imshow(combined_mask, cmap='jet', interpolation='nearest')
#         plt.title(f"Ground Truth Mask ({len(rle_parts)} Instances)")
#         plt.colorbar(label="Instance ID")
#         plt.axis('off')

#         plt.tight_layout()
#         plt.show()

# # --- Run Visualization ---
# visualize_generated_samples(num_samples=5)

In [5]:
!zip -r -q synthetic_dataset.zip /kaggle/working/synthetic_forgery_dataset

print("✅ Zipping complete.")
# Verify the file was created and show its size
!ls -lh synthetic_dataset.zip

✅ Zipping complete.
-rw-r--r-- 1 root root 8.9G Dec 30 11:17 synthetic_dataset.zip


In [6]:
import os

# --- PATHS ---
original_data_root = "/kaggle/input/data-final/data"
synthetic_data_root = "/kaggle/working/synthetic_forgery_dataset"

print("--- FILE COUNT SUMMARY ---\n")

# 1. Count Original Dataset
print("📁 Original Dataset:")
if os.path.exists(original_data_root):
    total_original = 0
    # List subdirectories (Blot-Gel, Microscopy, etc.)
    for folder in sorted(os.listdir(original_data_root)):
        folder_path = os.path.join(original_data_root, folder)
        if os.path.isdir(folder_path):
            # Count image files inside each subdirectory
            count = len([f for f in os.listdir(folder_path) if f.lower().endswith(('png', 'jpg', 'jpeg'))])
            print(f"  - {folder:<15}: {count} images")
            total_original += count
    print(f"  --------------------------")
    print(f"  Total Original Images: {total_original}\n")
else:
    print(f"  - Path not found: {original_data_root}\n")

# 2. Count Synthetic Dataset
print("📁 Synthetic Dataset (Generated):")
if os.path.exists(synthetic_data_root):
    # Count generated images
    img_dir = os.path.join(synthetic_data_root, "images")
    if os.path.exists(img_dir):
        img_count = len([f for f in os.listdir(img_dir) if f.endswith('.png')])
        print(f"  - images/: {img_count} forged images")
    else:
        print("  - images/ folder not found.")
        
    # Count generated masks
    mask_dir = os.path.join(synthetic_data_root, "masks")
    if os.path.exists(mask_dir):
        mask_count = len([f for f in os.listdir(mask_dir) if f.endswith('.npy')])
        print(f"  - masks/: {mask_count} RLE mask files")
    else:
        print("  - masks/ folder not found.")
else:
    print(f"  - Path not found: {synthetic_data_root}")

print("\n--- End of Summary ---")

--- FILE COUNT SUMMARY ---

📁 Original Dataset:
  - Path not found: /kaggle/input/data-final/data

📁 Synthetic Dataset (Generated):
  - images/: 19000 forged images
  - masks/: 19000 RLE mask files

--- End of Summary ---


In [7]:
# import shutil
# import os

# # Define the path to the directory you want to delete
# directory_path = '/kaggle/working/synthetic_forgery_dataset'

# # Ensure the path exists before attempting deletion to avoid FileNotFoundError
# if os.path.exists(directory_path) and os.path.isdir(directory_path):
#     try:
#         # Recursively remove the directory and all its contents
#         shutil.rmtree(directory_path)
#         print(f"Successfully deleted: {directory_path}")
#     except OSError as e:
#         print(f"Error deleting directory {directory_path}: {e}")
# else:
#     print(f"Directory not found or is not a directory: {directory_path}")
